# Sentence-BERT + SVM

Sentence-BERT (SBERT) is a pre-trained transformer, fine-tuned BERT on sentence-similarity tasks, that converts any sentence or paragraph into a fixed-length dense vector (typically 384 or 768 dimensions). Unlike TF-IDF, which produces sparse vectors with one dimension per word, SBERT produces dense vectors where each dimension is a learned semantic feature.

**Two sentences with similar meaning get similar vectors even if they share no words.**

This approach is a "baseline" because it uses a pre-trained transformer but does not adapt it to this specific task.

Model options:
- all-mpnet-base-v2 — the default strong choice as of 2026. 768-dim vectors.
- all-MiniLM-L6-v2 — smaller and faster. 384-dim. About 1% lower quality but 5x faster.
- BAAI/bge-large-en-v1.5 — top of the embedding leaderboards. 1024-dim. Slower to encode.

**Long documents**. SBERT models have a maximum input length (typically 384 or 512 tokens). Some TherapistQA patient questions are longer than this. The model will truncate them silently.

TherapistQA's label distribution is heavily skewed - using weighted F1 because it weights each class's F1 by its support (number of examples).

In [9]:
from sentence_transformers import SentenceTransformer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
import pandas as pd

# Load data
df = pd.read_csv('processed/therapistqa.csv')

# Split
X_train, X_test, y_train, y_test = train_test_split(
    df['patient_question'],
    df['dominant_distortion'],
    test_size=0.2,
    stratify=df['dominant_distortion'],
    random_state=42
)

# Load pre-trained SBERT model
# all-mpnet-base-v2 (the strongest general-purpose SBERT model as of 2026, produces 768-dimensional embeddings)
# all-MiniLM-L6-v2 (smaller and faster, 384-dim. About 1% lower quality but 5x faster.)
# BAAI/bge-large-en-v1.5 (top of the embedding leaderboards. 1024-dim. Slower to encode.)
model = SentenceTransformer('BAAI/bge-large-en-v1.5')

# Encode all documents into vectors
# This takes a few minutes on CPU, seconds on GPU
X_train_emb = model.encode(X_train.tolist(), show_progress_bar=True)
X_test_emb = model.encode(X_test.tolist(), show_progress_bar=True)

# Train SVM on the embeddings
svm = LinearSVC(C=1.0, class_weight='balanced', max_iter=5000)
svm.fit(X_train_emb, y_train)

# Evaluate
y_pred = svm.predict(X_test_emb)
print(classification_report(y_test, y_pred, digits=3))

Batches: 100%|██████████| 16/16 [00:33<00:00,  2.08s/it]


                         precision    recall  f1-score   support

All-or-nothing thinking      0.148     0.200     0.170        20
    Emotional Reasoning      0.154     0.148     0.151        27
        Fortune-telling      0.108     0.138     0.121        29
               Labeling      0.185     0.152     0.167        33
          Magnification      0.135     0.128     0.132        39
          Mental filter      0.140     0.292     0.189        24
           Mind Reading      0.340     0.333     0.337        48
          No Distortion      0.671     0.535     0.595       187
     Overgeneralization      0.281     0.188     0.225        48
        Personalization      0.211     0.267     0.235        30
      Should statements      0.139     0.238     0.175        21

               accuracy                          0.330       506
              macro avg      0.228     0.238     0.227       506
           weighted avg      0.375     0.330     0.346       506



/Users/ivagoluza/Documents/cognitive-distortioins-ml/dataset_preparation/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/ivagoluza/Documents/cognitive-distortioins-ml/dataset_preparation/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/ivagoluza/Documents/cognitive-distortioins-ml/dataset_preparation/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
